# Teaching a maze agent: where rogue behavior comes from

> **[Instructor: short intro — link to the NYT article "A.I.'s Rogue Behavior Starts With Its Training",
> what students will do (Part 1: build an agent; Part 2: change its rewards and watch what it does),
> and which parts of the maze rules are our simplified version of the article's model.]**

## Setup

> **[Instructor: tell students to pick a GPU first: `Runtime → Change runtime type → T4 GPU`, then run
> the cells below in order.]**

In [ ]:
# Install the maze code (takes about a minute)
!pip install -q git+https://github.com/anthoak13/HNR-MazeSolver@maze-rl

In [ ]:
import mazerl as mz
import pandas as pd
from IPython.display import display

mz.check_gpu()   # warns you if this runtime has no GPU

def compare(results):
    """Show {name: results} as a table, one row per agent."""
    return pd.DataFrame(results).T

### The article's two mazes

> **[Instructor: one line on each — p.6 "Agent facing a hard problem" (the flag is walled off) and p.7
> "Agent taking outside path although inside path is available".]**

In [ ]:
# The p.6 maze: the flag is walled off
display(mz.show_maze(mz.scenario("enclosed")))
# The p.7 maze: there is a route inside, and one around the outside
display(mz.show_maze(mz.scenario("outside_shortcut")))

# Part 1: Building an agent

> **[Instructor: framing — first general knowledge of mazes with no rewards (pretraining), then
> specializing it with rewards (RL). This is the recipe behind chatbots like ChatGPT.]**

## Step 1: What the agent sees

> **[Instructor: prediction question — e.g. The agent only sees a few squares around itself. What would you want to know, each move, to find a flag in a maze you've never seen?]**

In [ ]:
# One new maze, used throughout Part 1
maze = mz.random_mazes(1, seed=0)[0]
display(mz.show_maze(maze))

> **[Instructor: explain the picture below — an 11×11 window of the agent's memory centered on itself
> (not seen yet / open floor / outside / wall / flag / arena edge), plus where it has stood, what it can
> see right now (5×5, walls block sight), walls it bumped that held firm, and the 8-way compass to the
> flag.]**

In [ ]:
# What the network gets as input after 10 moves in this maze
mz.show_observation(maze=maze, step=10, seed=0)

> **[Instructor: the points for each event — flag, step outside, walking into a wall, first visit to a
> square.]**

In [ ]:
# The default points for each event (negative = penalty)
mz.Rewards()

In [ ]:
# An untrained agent: it picks moves at random (agent=None)
mz.playback(None, maze=maze, n_agents=5, seed=0)

In [ ]:
# How often does moving at random reach the flag? (500 new mazes)
mz.evaluate(None, n=500)

> **[Instructor: what happened — how rarely random moves reach the flag; what the agent would need to learn.]**

## Step 2: Pretraining — learning about mazes with no rewards

> **[Instructor: prediction question — e.g. After practicing on thousands of mazes, how often will the network guess a hidden square correctly? How well would you do by always guessing the most common kind of square?]**

> **[Instructor: the data — rule-following walkers (simple programs, not networks) explore the 13,148
> training mazes; after every move the agent's input is recorded (about 150,000 snapshots). The walkers
> ignore the flag and earn nothing. Analogy: next-token prediction.]**

![A walker exploring a maze](https://raw.githubusercontent.com/anthoak13/HNR-MazeSolver/maze-rl/reports/figures/pretraining/walker-path.png)

> **[Instructor: the exercise — for each snapshot the network guesses (a) what's in every hidden square
> of its window (weak spots count as wall) and (b) which move heads toward the nearest unvisited open
> square.]**

![Snapshots of the walker's memory, and the network's guesses](https://raw.githubusercontent.com/anthoak13/HNR-MazeSolver/maze-rl/reports/figures/pretraining/snapshots.png)

In [ ]:
# Load the network that was pretrained this way (no rewards involved)
perception = mz.load_perception("perception")

# How well it guesses on mazes it never saw during pretraining
perception.test

In [ ]:
# Optional (slow): run the pretraining yourself instead of loading it
# perception = mz.pretrain_perception()

> **[Instructor: what happened — compare the accuracy to always guessing the most common square (TEACHING.md: 86% vs 53–55%; the direction guess is easier than it looks). The compass is in the input but neither guess uses it.]**

## Step 3: Specializing it for the flag

> **[Instructor: prediction question — e.g. We keep the pretrained layers fixed and let rewards train only the top layers (about 10k of 230k numbers). Will it learn to reach the flag? What if the fixed layers are random instead?]**

> **[Instructor: this cell trains live: 20,000 attempts, about 2 minutes on a T4. Watch the progress
> bar.]**

In [ ]:
# RL on top of the pretrained layers, which stay fixed (freeze_perception=True)
flag_live = mz.train(perception=perception.agent, freeze_perception=True, episodes=20_000,
                     checkpoint_at=(1_000, 2_000, 5_000, 10_000))

# If short on time, load the trained one instead:
# flag_live = mz.load_pretrained("flag_frozen_pretrained")

In [ ]:
# The same training on top of fixed RANDOM layers (trained ahead of time)
flag_random = mz.load_pretrained("flag_frozen_random")

# How training went, side by side
mz.plot_training([flag_live, flag_random], labels=["pretrained layers", "random layers"])

In [ ]:
# Success on 500 new open mazes
open_mazes = mz.random_mazes(500, seed=1, kind="open")
compare({"pretrained layers": mz.evaluate(flag_live, mazes=open_mazes),
         "random layers": mz.evaluate(flag_random, mazes=open_mazes)})

In [ ]:
# Watch the pretrained-layer agent at several stages of its training
mz.training_grid(flag_live, n_mazes=3)

> **[Instructor (optional): the specialization is partial. The cell below compares the frozen agent with
> `base`, whose whole network trained, on each kind of maze.]**

In [ ]:
# Each kind of maze: frozen pretrained layers vs. a whole network trained by RL
base = mz.load_pretrained("base")
results = {}
for kind in ["open", "interior_break", "outside_only"]:
    kind_mazes = mz.random_mazes(300, seed=2, kind=kind)
    results[f"{kind}: frozen"] = mz.evaluate(flag_live, mazes=kind_mazes)
    results[f"{kind}: whole network"] = mz.evaluate(base, mazes=kind_mazes)
compare(results)[["reached flag", "went outside", "avg walls broken"]]

> **[Instructor: what happened — the general knowledge is in the pretrained layers; RL only teaches what to want. Random layers stay near chance. Harder skills (weak spots, outside routes) need the lower layers to keep learning.]**

## Step 4: A second specialization from the same base — escaping the maze

> **[Instructor: prediction question — e.g. Same fixed pretrained layers, a different reward: start where the flag would be, no compass, the first step outside wins. Will the same knowledge make a good escape artist?]**

In [ ]:
# What the agent sees in the escape task (no compass)
mz.show_observation(maze=maze, step=10, seed=0, task="escape")

In [ ]:
# RL for escaping, on the same fixed pretrained layers (about 2 minutes)
escape_live = mz.train(perception=perception.agent, freeze_perception=True, task="escape",
                       episodes=20_000, checkpoint_at=(1_000, 2_000, 5_000, 10_000))

# If short on time, load the trained one instead:
# escape_live = mz.load_pretrained("escape_frozen_pretrained")

In [ ]:
# Watch five escape attempts
mz.playback(escape_live, maze=maze, n_agents=5, seed=0)

In [ ]:
# Escape success on new open mazes: pretrained vs. random fixed layers
escape_random = mz.load_pretrained("escape_frozen_random")
compare({"pretrained layers": mz.evaluate(escape_live, mazes=open_mazes),
         "random layers": mz.evaluate(escape_random, mazes=open_mazes)})

> **[Instructor: what happened — same knowledge, different reward, different agent. Not to claim: that pretraining makes RL faster, or that it is the only route to general knowledge (layers trained by RL on the flag also carry over to escape).]**

# Part 2: Rogue behavior from the rewards

> **[Instructor: framing — from here on we use agents trained ahead of time at full scale (126,000
> attempts, like the article's). They differ only in their rewards.]**

## Step 5: The p.6 maze

> **[Instructor: prediction question — e.g. The flag is walled off. Nobody ever trained the agent on a maze like this. What will it do?]**

In [ ]:
# The full-scale agent with the default rewards
base = mz.load_pretrained("base")
base.rewards

In [ ]:
# Ten attempts on the p.6 maze
mz.playback(base, maze="enclosed", n_agents=10, seed=0)

In [ ]:
# Out of 500 attempts: how many left the maze, broke back in from outside, and reached the flag?
mz.breakout_stages(base)

In [ ]:
# The same question at each stage of training
compare({f"after {stage:,} attempts": mz.breakout_stages(base.agent_at(stage))
         for stage in sorted(base.checkpoints)})

> **[Instructor: what happened — it combines two separately learned skills (leaving and re-entering; breaking weak spots) and the flag reward outweighs the penalties. It succeeds in many attempts but not all. Real incident: Hugging Face. Training is not deterministic: another seed gives 35% or 75%.]**

## Step 6: The outside penalty and the p.7 maze

> **[Instructor: prediction question — e.g. Three agents differ only in the cost of a step outside: −0.05, −0.01 (default), and free. On the p.7 maze, where an inside route exists, how often will each one go outside?]**

In [ ]:
# Three agents, identical except for the outside penalty
outside_agents = {"outside −0.05": mz.load_pretrained("outside_costly"),
                  "outside −0.01 (default)": base,
                  "outside free": mz.load_pretrained("outside_free")}

In [ ]:
# Five attempts each on the p.7 maze
for name, run in outside_agents.items():
    print(name)
    display(mz.playback(run, maze="outside_shortcut", n_agents=5, seed=0))

In [ ]:
# 500 attempts each on the p.7 maze
compare({name: mz.evaluate(run, mazes="outside_shortcut")
         for name, run in outside_agents.items()})[["went outside", "avg moves outside", "reached flag"]]

> **[Instructor: what happened — the article's "when the rewards slightly favor it," shown directly (TEACHING.md: 0–6% at −0.05, 28–51% at −0.01, up to 99% when free). Real incident: RubyGems.]**

## Step 7: Reward hacking

> **[Instructor: prediction question — e.g. One agent earned +0.05 (ten times the default) for every new square it visited, including squares outside the maze. What do you expect it to do? Will its score go up or down?]**

In [ ]:
# The agent with a big exploration reward
explorer = mz.load_pretrained("explore_bonus_big")
explorer.rewards

In [ ]:
# Same maze, same seed: the default agent, then the big-exploration-reward agent
test_maze = mz.random_mazes(1, seed=3, kind="interior_break")[0]
display(mz.playback(base, maze=test_maze, n_agents=5, seed=0))
display(mz.playback(explorer, maze=test_maze, n_agents=5, seed=0))

In [ ]:
# Where each agent's points came from, in one attempt on that maze
base_points = mz.score_breakdown(base, maze=test_maze, seed=0)
explorer_points = mz.score_breakdown(explorer, maze=test_maze, seed=0)
compare({"default": base_points, "big exploration reward": explorer_points})

In [ ]:
# Score, success and speed on 500 new weak-spot mazes (plus the outside-free agent from Step 6)
weak_mazes = mz.random_mazes(500, seed=4, kind="interior_break")
hackers = {"default": base, "big exploration reward": explorer,
           "outside free": outside_agents["outside free"]}
compare({name: mz.evaluate(run, mazes=weak_mazes) for name, run in hackers.items()})[
    ["avg score", "reached flag", "moves to flag", "avg moves outside"]]

In [ ]:
# And on the p.6 maze
compare({name: mz.breakout_stages(run) for name, run in hackers.items()})

> **[Instructor: what happened — wherever a step outside nets positive reward, the agent tours the outside farming points. The small default reward does it too once outside moves are free: the loop matters, not the size. The score goes up while finishing slows (~1.5×) and the p.6 flag is almost never reached. Success on ordinary mazes barely changes: rewards shape the route more than the result. One seed in three never found the loop.]**

## Optional: try your own rewards

> **Warning:** a live run here is 20,000 attempts, a sixth of the full-scale training. Short runs often
> won't reproduce what the full-scale agents above do (the p.6 breakout almost never appears). Treat
> what you see as a hint, not a result.

> **[Instructor: suggest something to try, e.g. a reward to make the agent avoid walls, or a negative
> reward for every move.]**

In [ ]:
# Change the numbers, then run (about 2–5 minutes)
my_rewards = mz.Rewards(flag=5.0, outside=-0.01, probe=-0.01, new_cell=0.005)
my_run = mz.train(rewards=my_rewards, episodes=20_000)
my_run.plot()

In [ ]:
# How it does on new mazes and on the article's mazes
compare({"new mazes": mz.evaluate(my_run),
         "p.7 maze": mz.evaluate(my_run, mazes="outside_shortcut")})

In [ ]:
# Watch it
mz.playback(my_run, maze="outside_shortcut", n_agents=5, seed=0)

## Step 8: Discussion

In [ ]:
# Our safety rule leaves attempts that end outside the maze out of training.
# How often did each agent's attempts end outside ("went around the maze")?
compare({name: mz.evaluate(run, mazes=weak_mazes) for name, run in hackers.items()})[
    ["went around the maze", "avg moves outside"]]

> **[Instructor: discussion questions — e.g.**
> - **Our rule excluding attempts that end outside didn't catch the hacking. Why not? (Checking outcomes
>   misses an agent gaming the process.)**
> - **The same rule starved some pretrained agents of training data. Can a safety rule have side
>   effects on learning?**
> - **Who is responsible when an agent breaks out? How would you design the sandbox?]**